In [ ]:
!pip install -U pageindex openai python-dotenv

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

# NO hardcoded keys — load from .env only
PAGEINDEX_API_KEY = os.getenv('PAGEINDEX_API_KEY', '')
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY')

print('PageIndex:', 'v' if PAGEINDEX_API_KEY else 'Missing — add to .env')
print('OpenAI:   ', 'v' if OPENAI_API_KEY else 'Missing — add to .env')

In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print('Clients ready')

In [ ]:
PDF_PATH = './sample_document.pdf'

if not PAGEINDEX_API_KEY:
    print('Add PAGEINDEX_API_KEY to .env before uploading')
else:
    print(f'Uploading: {PDF_PATH}')
    result = pi_client.submit_document(PDF_PATH)
    doc_id = result['doc_id']
    print(f'Uploaded — doc_id={doc_id}')

In [ ]:
if PAGEINDEX_API_KEY and 'doc_id' in locals():
    while True:
        try:
            status = pi_client.get_document(doc_id).get('status')
            print(f'Status: {status}')
            if status == 'completed':
                print('Tree index ready!')
                break
            elif status == 'failed':
                print('Failed — check PDF format')
                break
            time.sleep(5)
        except Exception as e:
            print(f'Check error: {e}')
            time.sleep(5)
else:
    print('Skip — set API key and upload first')

In [ ]:
if PAGEINDEX_API_KEY and 'doc_id' in locals():
    try:
        tree = pi_client.get_tree(doc_id, node_summary=True).get('result', [])
        print(f'Top sections: {len(tree)}')
        print('First node:', json.dumps(tree[0] if tree else {}, indent=2))
    except Exception as e:
        print(f'Tree error: {e}')
else:
    print('Skip — upload + build index first')

In [ ]:
def print_tree(nodes, indent=0):
    for node in nodes:
        p = node.get('page_index', '?')
        print(f"{'  '*indent}└─ [{node['node_id']}] {node['title']} (p.{p})")
        if node.get('nodes'): print_tree(node['nodes'], indent+1)

if PAGEINDEX_API_KEY and 'tree' in locals():
    print('Full Document Structure:\n'); print_tree(tree)
else:
    print('Skip — tree not built')

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get('nodes'): total += count_nodes(n['nodes'])
    return total

if 'tree' in locals():
    print(f'Total nodes: {count_nodes(tree)}')
else:
    print('Skip')

In [ ]:
def llm_tree_search(query, tree, model='gpt-4o'):
    compressed = [{'node_id':n['node_id'],'title':n['title'],'page':n.get('page_index','?'),'summary':n.get('text','')[:150],
        'children': [{'node_id':c['node_id'],'title':c['title']} for c in n.get('nodes',[])] if n.get('nodes') else None}
        for n in tree]
    prompt = f"Query: {query}\nDoc Tree: {json.dumps(compressed,indent=2)}\n\nReturn JSON only: {{\"thinking\":..., \"node_list\":[...]}}"
    resp = openai_client.chat.completions.create(model=model,messages=[{'role':'user','content':prompt}],response_format={'type':'json_object'})
    return json.loads(resp.choices[0].message.content)

if PAGEINDEX_API_KEY and OPENAI_API_KEY and 'tree' in locals():
    result = llm_tree_search('What is the syllabus?', tree)
    print('Reasoning:', result.get('thinking'))
    print('Nodes:', result.get('node_list'))
else:
    print('Skip — set both keys and build tree')

In [ ]:
def find_nodes_by_ids(tree, ids):
    found=[]
    for node in tree:
        if node['node_id'] in ids: found.append(node)
        if node.get('nodes'): found.extend(find_nodes_by_ids(node['nodes'], ids))
    return found

def generate_answer(query, nodes, model='gpt-4o'):
    if not nodes: return 'No sections found.'
    context = '\n\n---\n\n'.join([f"[Section: '{n['title']}' | p.{n.get('page_index','?')}]\n{n.get('text','')}" for n in nodes])
    prompt = f"Answer using ONLY context. Cite sections/pages.\nQuestion: {query}\n\nContext:\n{context}\n\nAnswer:"
    resp = openai_client.chat.completions.create(model=model, messages=[{'role':'user','content':prompt}])
    return resp.choices[0].message.content

if 'result' in locals() and 'tree' in locals():
    nodes = find_nodes_by_ids(tree, result.get('node_list', []))
    print('=== ANSWER ===\n', generate_answer('What is syllabus?', nodes))
else:
    print('Skip — complete earlier steps')